In [ ]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv("../data/tmdb_movies.csv")

df = df.dropna(subset=["overview", "budget", "revenue"])
df = df[(df["budget"] > 0) & (df["revenue"] > 0)]

df["roi"] = (df["revenue"] - df["budget"]) / df["budget"]

df = df[df["roi"] < df["roi"].quantile(0.99)]

good_df = df[df["roi"] >= 3].copy()

good_df = good_df.sample(n=min(300, len(good_df)), random_state=42)


output_path = "../data/movie_roi_finetune_good_only.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for _, row in good_df.iterrows():
        example = {
            "messages": [
                {
                    "role": "system",
                    "content": "You are a creative movie concept generator. Generate original movie overviews similar to high-ROI movies.",
                },
                {
                    "role": "user",
                    "content": "Generate a movie overview for a potential high-ROI movie.",
                },
                {"role": "assistant", "content": str(row["overview"]).strip()},
            ]
        }

        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Saved:", output_path)
print("Examples:", len(good_df))
print(good_df[["overview", "budget", "revenue", "roi"]].head())

Saved: ../data/movie_roi_finetune_good_only.jsonl
Examples: 300
                                               overview     budget    revenue  \
4786  Between the events of 'Saw' and 'Saw II', a si...   13000000  125319714   
3406  Ricky is a defiant young city kid who finds hi...    2500000   23900000   
3646  A dog goes on quest to discover his purpose in...   22000000  205000000   
2946  After surviving the Hunger Games, Katniss and ...  130000000  865011746   
2271  Johnny Knoxville, Bam Margera, Steve-O, Wee Ma...   20000000  117224271   

           roi  
4786  8.639978  
3406  8.560000  
3646  8.318182  
2946  5.653937  
2271  4.861214  


In [13]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI()

training_file = client.files.create(
    file=open("../data/movie_roi_finetune_good_only.jsonl", "rb"), purpose="fine-tune"
)

print(training_file.id)

file-C6AhRB5FYFd3EX2bXefaZm


In [14]:
fine_tune_job = client.fine_tuning.jobs.create(
    training_file=training_file.id, model="gpt-3.5-turbo"
)

print(fine_tune_job.id)

ftjob-267x9napCX2evX2dgeNhoEV2


In [19]:
job = client.fine_tuning.jobs.retrieve(fine_tune_job.id)

print("Status:", job.status)
print("Fine-tuned model:", job.fine_tuned_model)

Status: succeeded
Fine-tuned model: ft:gpt-3.5-turbo-0125:datify::DXzk8rSS


In [20]:
response = client.chat.completions.create(
    model=job.fine_tuned_model,
    messages=[
        {"role": "system", "content": "You are a creative movie concept generator."},
        {
            "role": "user",
            "content": "Generate a movie overview for ROI category: high performer.",
        },
    ],
    temperature=0.9,
)

print(response.choices[0].message.content)

A comedy that follows an ex-con who lands a position at a school that sits over the spot where money from one of his earlier robberies was stashed.


In [25]:
response = client.chat.completions.create(
    model=job.fine_tuned_model,
    messages=[
        {"role": "system", "content": "You are a creative movie concept evaluator."},
        {
            "role": "user",
            "content": "Evaluate this overview from 0 to 100. Overview:'A comedy that follows an ex-con who lands a position at a school that sits over the spot where money from one of his earlier robberies was stashed.'",
        },
    ],
    temperature=0.9,
)

print(response.choices[0].message.content)

40
